# JSE Top 40: EDA, experimentation, optimisation and longitudinal review

Run the cells in order. The study contains **40 economic groups represented by 41 securities** at each of 2025-06-30, 2025-12-31 and 2026-06-30. INL and INP are the two Investec DLC securities. Membership is date-specific.

This notebook makes the experiment trail visible: all declared parameter combinations, invalid exclusions, both quality scores, selected parameters, all-date dendrograms, stable/shifted instruments, descriptive geometric screens, sector context and SHAP. It is a descriptive screening study, with unresolved historical source certification.

For review, set `REUSE_COMPLETED_RUN` in the setup cell to a completed ID such as `paper-exhaustive-k8-01`. This verifies the archive and displays saved tables/figures without refitting. The executed review copy explicitly records that read-only mode.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from paper_experiments import PaperExperiment, SavedPaperExperiment
# Set to a completed run ID to review saved results without refitting anything.
# Leave None for a fresh full experiment (about 10 minutes on the development PC).
REUSE_COMPLETED_RUN = os.environ.get("JSE_PAPER_REUSE_RUN") or None
study = (SavedPaperExperiment(REUSE_COMPLETED_RUN) if REUSE_COMPLETED_RUN
         else PaperExperiment(run_id=os.environ.get("JSE_PAPER_RUN_ID")))
print(study.out)

## 1. EDA before modelling

Inspect population, source evidence, flagged missing cases, common-scale distributions and correlations. June 2025 preprocessing is frozen for later dates. The data are overlapping historical populations, not 123 independent firms.

In [ ]:
display(study.explore())
study.show_figure("eda/population_and_sectors.png")
display(study.table("eda/missingness.csv"))
study.show_figure("eda/feature_distributions.png")

In [ ]:
for date in study.config["dates"]:
    study.show_figure(f"eda/correlation_{date}.png")
study.show_figure("eda/membership_presence.png")
display(study.table("eda/evidence.csv"))

## 2. Declare and run all valid combinations in the finite grid

The exact grid is in `configs/paper_experiments.json`. **86 representations x 38 clustering settings = 3,268 configurations; five seeds = 16,340 selection fits**. A representation is fitted once per seed and reused across clustering settings.

- PCA: two components or enough for 95% variance (potentially more than two).
- t-SNE: perplexities 5/10/20; learning rates 10/auto; 300/1000 iterations; Euclidean/Manhattan/cosine input distances.
- UMAP: neighbours 3/5/10/20; min_dist 0/0.01/0.1/0.5; Euclidean/Manhattan/cosine input distances.
- Hierarchical: single/complete/average with Euclidean/Manhattan/Chebyshev/cosine; Ward with Euclidean only. k=8.
- K-Means: k=8, n_init=20, Euclidean squared-distance objective.
- DBSCAN: eps 0.3/0.5/0.8/1.2; min_samples 3/5; Euclidean/Manhattan/cosine after axis standardization. Actual cluster/noise counts remain explicit.

Ward plus non-Euclidean distances is invalid and excluded explicitly. k is fixed at 8 for the paper's hierarchical/K-Means comparisons. Other k values are later diagnostics. This is exhaustive **within the declared finite grid**, not every possible parameter value or a guaranteed global optimum.

In [ ]:
display(Markdown("```json\n" + json.dumps(study.config, indent=2) + "\n```"))
study.experiment()
display(study.table("search/candidate_summary.csv").head(10))

## 3. Select with BOTH silhouette and Davies-Bouldin

On June 2025, require valid scores and full coverage on every seed, and eight clusters for the overall paper comparison. Minimise the sum of ranks: mean silhouette descending and mean DB ascending. Tie-break by seed ARI, feature silhouette and deterministic candidate ID.

The **primary criterion uses common June-scaled financial features** to compare the same geometry across reductions. A separate embedding criterion shows how the apparent winner changes with the score space. Sector ARI/NMI never tune the model. DB uses Euclidean centroids; score geometry differs from some fitting distances. Inspect individual metric champions and Pareto sets, not only the joint winner. A family without an eligible candidate is reported rather than silently removed.

Settings are frozen from June 2025 and reused at later dates. Because later dates have already been examined in this project, they are descriptive follow-ups, not untouched validation sets.

In [ ]:
display(study.select_models())
display(study.table("search/family_best_models.csv"))
study.show_figure("search/search_tradeoffs.png")

In [ ]:
display(study.table("search/distance_linkage_inventory.csv"))
for role in ["common_features_k8", "embedding_k8"]:
    ranking = study.table(f"search/ranking_{role}.csv")
    display(Markdown(f"### {role}: top joint candidates"))
    display(ranking.head(10))
    geometry = "features" if role == "common_features_k8" else "embedding"
    display(ranking.sort_values(f"silhouette_{geometry}_mean", ascending=False).head(1))
    display(ranking.sort_values(f"davies_bouldin_{geometry}_mean").head(1))

## 4. Results at each date with frozen parameters

The representative fit is seed 42. Save all five seeds and pairwise stability. Each selected hierarchical model gets a dendrogram built from its **actual fitted tree**; non-hierarchical models do not get an invented dendrogram. PCA95 trees use all retained dimensions, even though maps show the first two coordinates.

In [ ]:
display(study.fit_results())
display(study.table("selected_seed_stability.csv"))
display(study.table("cluster_surrogate_fidelity.csv"))

In [ ]:
# Show the globally selected common-feature model's complete dendrogram series.
# If a non-hierarchical model wins another grid, use the saved baseline tree series.
CID = study.choices["common_features_k8"]
if not (study.out / "models" / CID / study.config["dates"][0] / "dendrogram.png").exists():
    CID = study.choices["baseline"]
for date in study.config["dates"]:
    study.show_figure(f"models/{CID}/{date}/dendrogram.png")
study.show_figure(f"models/{CID}/k_diagnostics.png")

In [ ]:
# View the primary selected model's membership and feature profile at each date.
PRIMARY = study.choices["common_features_k8"]
for date in study.config["dates"]:
    study.show_figure(f"models/{PRIMARY}/{date}/cluster_map.png")
    study.show_figure(f"models/{PRIMARY}/{date}/feature_profile.png")

## 5. Stable/shifted instruments, sector divergence and descriptive outlier statistics

Align arbitrary labels with Hungarian maximum Jaccard on common members. Sorted labels and a deterministic solver make the procedure reproducible; zero-overlap matches get fresh IDs. Compare with maximum-overlap alignment and direct baseline-to-current alignment. Only all-three-date members receive a three-date stable/shifted status. Entries and exits are separate events.

The individual geometric statistic uses the same-date mean and unbiased covariance:

$$D_i^2=(z_i-\bar{z})^{T}S^{-1}(z_i-\bar{z}),\qquad p=2.$$

**There is no n multiplier.** Its descriptive reference is

$$c_F=\frac{p(n-1)}{n-p}\,F_{0.95}(p,n-p).$$

This mean-T-squared F formula is not the individual-distance null calibration. We also record an internal Beta reference as a sensitivity. Singular/ill-conditioned covariance means no screen is estimable. Data-adaptive t-SNE/UMAP coordinates do not justify a confirmed significance claim.

A representative-seed screen flag supported by at least 3/5 estimable seeds is labelled multi-seed-supported. It is still a **candidate**, not a confirmed outlier. Three-date flags distinguish emerging, persistent, transient, resolving and recurring histories; missing membership/screen means ineligible.

A singleton, geometric extreme and sector-divergent firm are separate cases. Sector divergence compares a firm with the leave-one-out majority sector of its cluster; ties and singletons remain unresolved. An additional maximum-overlap cluster/sector mapping is explicitly contextual. The legacy sectors are not independently verified historic ground truth.

In [ ]:
display(study.longitudinal_review())
review = study.table("longitudinal_summary.csv")
PRIMARY_FOLDER = f"models/{PRIMARY}"
if (study.out / PRIMARY_FOLDER / "company_cluster_histories.csv").exists():
    display(study.table(f"{PRIMARY_FOLDER}/stable_instruments.csv"))
    display(study.table(f"{PRIMARY_FOLDER}/shifted_instruments.csv"))
    display(study.table(f"{PRIMARY_FOLDER}/membership_transitions.csv").query("membership_event != 'continuing'"))
    study.show_figure(f"{PRIMARY_FOLDER}/cluster_histories.png")
    study.show_figure(f"{PRIMARY_FOLDER}/jaccard_transition_heatmaps.png")

In [ ]:
if (study.out / PRIMARY_FOLDER / "company_case_summary.csv").exists():
    cases = study.table(f"{PRIMARY_FOLDER}/company_case_summary.csv")
    display(cases)
    display(study.table(f"{PRIMARY_FOLDER}/outlier_screen_histories.csv"))
    for date in study.config["dates"]:
        study.show_figure(f"{PRIMARY_FOLDER}/{date}/outlier_screen.png")
        display(study.table(f"{PRIMARY_FOLDER}/{date}/sector_context.csv"))

## 6. Explain attributes, with the correct SHAP target

Three distinct explanations are saved:

1. Each date's classifier surrogate: associations with cluster membership, with in-sample/OOF fidelity and raw-margin additivity checks.
2. A separate regression surrogate of the geometric distance: associations with the screening score itself, with train/OOF R^2 and MAE. Cluster SHAP alone does not explain the Hotelling distance.
3. A **frozen 2025 classifier** evaluated at each continuing firm's 2025 and later feature vector. Destination-versus-source class SHAP contrasts are differenced within that same model. They sum to the margin-contrast change. This is an attribute-change description, not causal proof of why independently refitted clusters shifted. Unmatched destinations without a 2025 class are explicitly not estimable.

The instrument-level case report includes top SHAP attributes, actual feature changes, deviations from population medians, imputation flags and sector context. Stable-class zero contrasts do not imply unchanged finances. The case plots and all ten feature contributions are saved, rather than reporting only selected favourable examples.

In [ ]:
if (study.out / PRIMARY_FOLDER / "case_review.md").exists():
    display(Markdown((study.out / PRIMARY_FOLDER / "case_review.md").read_text(encoding="utf-8")))
    display(study.table(f"{PRIMARY_FOLDER}/frozen_2025_shift_checks.csv"))
    for date in study.config["dates"]:
        study.show_figure(f"{PRIMARY_FOLDER}/{date}/shap_global.png")
        display(Markdown("```json\n" + (study.out / PRIMARY_FOLDER / date / "screen_surrogate_fidelity.json").read_text() + "\n```"))

In [ ]:
# Change TICKER and VIEW_DATE to inspect an individual case. No refitting occurs.
TICKER = "NPN"
VIEW_DATE = "2026-06-30"
for name in [f"{VIEW_DATE}_{TICKER}.png", f"shift_{VIEW_DATE}_{TICKER}.png"]:
    image = study.out / PRIMARY_FOLDER / "cases" / name
    if image.exists(): study.show_figure(str(image.relative_to(study.out)))
if (study.out / PRIMARY_FOLDER / "company_feature_explanations.csv").exists():
    display(study.table(f"{PRIMARY_FOLDER}/company_feature_explanations.csv").query("ticker == @TICKER and decision_date == @VIEW_DATE"))

## 7. Validate and archive

Only the final stage marks this run completed. The registry, configuration, source snapshot, seeds, all grid trials, selected models, flags and explanation arrays remain auditable. No baseline is promoted automatically. Historical share counts, market capitalisations, sector history and vendor series still need source-backed validation for submission-grade findings.

In [ ]:
study.finish()
display(Markdown((study.out / "paper_results.md").read_text(encoding="utf-8")))
print("Saved complete run:", study.out)